# 0616ver_4 報告

## 報告主軸

老師上次看到的版本約是 `0609ver_3`，當時主要還在 template / schema / service contract。  
這次 `0616ver_4` 的重點是推進到可執行 service 流程：

```text
sensor_1min / sensor_3min
-> MonitoringWorker 定時查 DB
-> threshold 判斷 warning / fault
-> mapping 對應 cause_id / response_id
-> alert_event 回寫 Database/versionB
-> batch_station_status 更新
-> FutureService 回寫 future_prediction_result
```

這版不做 FastAPI、不走 HTTP endpoint、不改 UI 前端、不重設 DB schema。


In [ ]:
from pathlib import Path
import json
import os
import sys

# 建議把 Notebook 放在 少榆0616ver_4 根目錄執行。
PROJECT_DIR = Path.cwd()

if not (PROJECT_DIR / "webservices").exists():
    candidates = list(Path.cwd().glob("**/少榆0616ver_4"))
    if candidates:
        PROJECT_DIR = candidates[0]

print("PROJECT_DIR =", PROJECT_DIR)
print("webservices exists:", (PROJECT_DIR / "webservices").exists())

def show_file(path, start=1, end=None, keyword=None):
    p = PROJECT_DIR / path
    print("=" * 90)
    print(path)
    print("=" * 90)
    if not p.exists():
        print("找不到檔案：", p)
        return

    lines = p.read_text(encoding="utf-8", errors="replace").splitlines()

    if keyword:
        hits = [i for i, line in enumerate(lines, 1) if keyword in line]
        if not hits:
            print("找不到 keyword:", keyword)
            return
        start = max(1, hits[0] - 6)
        end = min(len(lines), hits[0] + 18)

    if end is None:
        end = min(len(lines), start + 60)

    for i, line in enumerate(lines[start-1:end], start):
        print(f"{i:04d}: {line}")

def show_json(path, max_chars=2500):
    p = PROJECT_DIR / path
    data = json.loads(p.read_text(encoding="utf-8"))
    text = json.dumps(data, ensure_ascii=False, indent=2)
    print(text[:max_chars])
    if len(text) > max_chars:
        print("\n...省略，報告時可打開完整檔案")


## 1. 如何接 Database/versionB

### 要開檔案
`webservices/integration_adapter/database_versionb_adapter.py`

### 主旨
集中處理 `Database/versionB` import path，讓少榆端直接使用 DB function。

### 作法
不走 HTTP endpoint，也不開 FastAPI；直接 import `Database/versionB` 裡的 `db_sensor`、`db_alert`、`db_status`、`db_future`。


In [ ]:
show_file("webservices/integration_adapter/database_versionb_adapter.py", start=1, end=90)


## 2. MonitoringWorker：老師說的 Timer / 背景檢查

### 要開檔案
`webservices/monitoring_worker/monitoring_worker.py`

### 主旨
負責定期查 DB、判斷異常、觸發 alert 回寫。

### 作法
老師上次問「alert event 是誰產生？」  
這版由 `MonitoringWorker` 產生。它查 `sensor_1min / sensor_3min`，判斷異常後寫回 DB。


In [ ]:
show_file("webservices/monitoring_worker/monitoring_worker.py", start=1, end=150)


## 3. 1Hz 改成 sensor_1min / sensor_3min

### 要開檔案
`webservices/monitoring_worker/monitoring_worker.py`

### 相關檔案
`schema/sensor_1min.schema.json`  
`schema/sensor_3min.schema.json`  
`csv_templates/sensor_1min_template.csv`  
`csv_templates/sensor_3min_template.csv`

### 作法
舊版的 1Hz 已經改掉。現在對齊 GitHub DB 的 `sensor_1min` / `sensor_3min`，並透過 `db_sensor.query_sensor_1min` / `query_sensor_3min` 查資料。


In [ ]:
show_file("webservices/monitoring_worker/monitoring_worker.py", keyword="query_sensor_1min")


## 4. Threshold 與 Mapping：從數值異常到原因對策

### 要開檔案
`rules/sensor_thresholds.json`  
`rules/sensor_event_mapping.json`  
`webservices/monitoring_worker/threshold_evaluator.py`  
`webservices/monitoring_worker/detection_mapping.py`

### 主旨
`threshold` 判斷 warning / fault；`mapping` 把異常轉成 cause_id / response_id。

### 作法
這版把「數值門檻」和「原因對策」分開。  
例如某個 sensor 超過 fault，會對應到具體 issue_state、cause_id、response_id，後續可寫入 alert link table。


In [ ]:
print("sensor_thresholds.json")
show_json("rules/sensor_thresholds.json", max_chars=1800)

print("\n" + "="*90 + "\nsensor_event_mapping.json")
show_json("rules/sensor_event_mapping.json", max_chars=2500)


## 5. Alert Event 回寫 Database

### 要開檔案
`webservices/monitoring_worker/alert_event_writer.py`

### 主旨
把異常偵測結果寫回 DB。

### 會寫入
`alert_event`  
`alert_cause_link`  
`alert_response_link`

### 作法
這裡就是上次說的「alert event 要回寫 database」。  
目前不是空 template，而是呼叫 `db_alert.insert_alert_event()`、`link_alert_cause()`、`link_alert_response()`。


In [ ]:
show_file("webservices/monitoring_worker/alert_event_writer.py", start=1, end=160)


## 6. batch_station_status 更新

### 要開檔案
`webservices/monitoring_worker/batch_station_status_writer.py`

### 主旨
把 alert 結果同步成 station / component 狀態，支援 Manager UI 查詢。

### 作法
我不是只寫 alert_event，也會更新 `batch_station_status`。  
這樣 UI 可以知道哪個 station、哪個 component 目前是 warning / fault。


In [ ]:
show_file("webservices/monitoring_worker/batch_station_status_writer.py", start=1, end=130)


## 7. FutureService 回寫 future_prediction_result

### 要開檔案
`webservices/future_service/future_service.py`

### 主旨
產生未來品質預測結果，並寫回 `future_prediction_result`。

### 對齊欄位
`prediction_id`、`batch_id`、`station_id`、`prediction_time`、`predicted_ok_rate`、`predicted_ng_count`、`quality_score`、`risk_level`、`model_input_source`、`created_at`

### 作法
上次提到預測資訊要讓 UI 使用，所以這版補了 `save_future_prediction_result()`，呼叫 `db_future.insert_future_prediction_result()`。


In [ ]:
show_file("webservices/future_service/future_service.py", start=1, end=160)


## 8. 防止重複 alert

### 要開檔案
`webservices/monitoring_worker/duplicate_alert_guard.py`

### 主旨
避免 Timer 每分鐘查資料時，同一筆異常被重複寫入。

### 作法
同一個 `batch_id + station_id + sensor_name + state + cause_id`，若短時間內已存在未確認 alert，就不重複寫入。


In [ ]:
show_file("webservices/monitoring_worker/duplicate_alert_guard.py", start=1, end=140)


## 9. DB 實測準備

### 要開檔案
`scripts/check_db_connection.py`  
`scripts/run_db_smoke_test.py`

### 主旨
準備連到 PostgreSQL 後的驗證流程。

### 作法
目前已準備 DB 連線檢查與 smoke test。  
尚未宣稱 PostgreSQL 端到端測試完成，因為需要等 DB 環境可用後執行。


In [ ]:
show_file("scripts/check_db_connection.py", start=1, end=120)
print("\n" + "="*90 + "\n")
show_file("scripts/run_db_smoke_test.py", start=1, end=160)


## 10. 實測時環境變數與指令

```powershell
$env:SPRAYLINE_PROJECT_ROOT="C:\path\to\Project-SprayLine-main"
$env:DB_HOST="localhost"
$env:DB_PORT="5432"
$env:DB_USER="postgres"
$env:DB_PASSWORD="你的PostgreSQL密碼"
$env:DB_NAME="sprayline"

$env:SPRAYLINE_MONITOR_STATIONS="Station_1"
$env:SPRAYLINE_MONITOR_LOOKBACK_MINUTES="10"
$env:SPRAYLINE_DUPLICATE_ALERT_SUPPRESSION_MINUTES="5"
```

執行：

```powershell
python scripts\check_db_connection.py
python scripts\run_db_smoke_test.py --write-test-data --station Station_1
```

實測成功後查：

```text
alert_event
alert_cause_link
alert_response_link
batch_station_status
future_prediction_result
```


## 11. UI 與 Batch summary 的說明

### UI 部分
 
負責讓資料寫回 DB，並整理 UI 後續可查的 DB function。

Manager UI 可查：
```python
get_manager_summary()
get_future_prediction_summary()
get_batch_detail()
get_latest_batches()
```

Engineer UI 可查：
```python
get_alert_detail()
get_alert_ui_card()
get_alert_causes()
get_alert_responses()
get_responses_for_cause()
get_batch_station_status()
```

### Batch summary
目前不新增 `batch_summary` table。  
目前做法是我這邊寫入 `alert_event`、`batch_station_status`、`future_prediction_result`，Manager summary 由 Database/versionB query function 查詢產生。  
若老師要求主動產生 batch_summary event，再和余宇承確認 schema 後新增。


## 12. 最後總結稿

這次的 `少榆0616ver_4` 是從上次 `0609ver_3` 的 template / schema 往 service 實作推進。

主要完成：

```text
1. 把 1Hz 改成 Database/versionB 的 sensor_1min / sensor_3min。
2. 補 MonitoringWorker，定時查 DB。
3. 用 threshold 判斷 warning / fault。
4. 用 mapping 對應 cause_id / response_id。
5. 產生 alert_event 並寫回 Database/versionB。
6. 建立 alert_cause_link / alert_response_link。
7. 更新 batch_station_status。
8. FutureService 可以寫回 future_prediction_result。
9. 補 duplicate alert suppression，避免同一異常重複寫入。
10. 補 DB 連線測試和 smoke test，等 DB 環境可用後可做端到端實測。
```

目前還差 PostgreSQL 實際端到端測試，這部分我已經把腳本和流程準備好了。
